# Qualcomm S24 ASR Benchmark - Google Colab

Notebook này chạy đúng benchmark **50 mẫu x 3 model** bằng runner đã kiểm thử:

- Whisper Tiny, Whisper Small, PhoWhisper Base.
- Neural encoder/decoder chạy trên NPU của exact hosted **Samsung Galaxy S24** qua Qualcomm AI Hub.
- Thu đủ WER, regional/noise/code-switch, latency và Peak RAM.
- Artifact, profile, checkpoint và kết quả lưu trên Google Drive để runtime ngắt vẫn resume được.
- Hugging Face và dataset cache dùng ổ local Colab nhanh hơn.
- Microbatch 4 để giảm số job; runner tự chia 2/1 nếu payload lớn không phù hợp.

**Cách chạy:** thêm `QAI_HUB_API_TOKEN` vào Colab Secrets nếu có, sau đó chọn **Runtime > Run all**. GPU không làm Qualcomm NPU nhanh hơn; CPU High-RAM là đủ. Nếu Colab ngắt phiên, mở lại notebook và Run all để tiếp tục checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/tanphong-sudo/qai_s24_benchmark.git'
REPO_DIR = Path('/content/qai_s24_benchmark')
DRIVE_ROOT = Path('/content/drive/MyDrive/qai_asr_s24_benchmark')
LOCAL_CACHE_ROOT = Path('/content/qai_s24_fast_cache')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)))
    return subprocess.run(command, cwd=cwd, env=env, check=True)

if (REPO_DIR / '.git').exists():
    print('Reusing repository already prepared in this Colab runtime.')
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not the benchmark repository')
else:
    run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_DIR)])

# Keep this one-file notebook fast even before its companion runner update
# reaches GitHub. Replacements are idempotent and change only batching/cache paths.
runner_path = REPO_DIR / 'run_benchmark.py'
runner_source = runner_path.read_text(encoding='utf-8')
colab_optimizations = {
    'BENCHMARK_N = 100            # 100 samples for each benchmark category': 'BENCHMARK_N = 50             # 50 samples for each benchmark category',
    'VIMD_BENCHMARK_REGION_TARGETS = {"North": 34, "Central": 33, "South": 33}': 'VIMD_BENCHMARK_REGION_TARGETS = {"North": 17, "Central": 17, "South": 16}',
    'HUB_MICROBATCH = 2': 'HUB_MICROBATCH = max(1, int(os.environ.get("QAI_HUB_MICROBATCH", "2")))',
    'HF_HOME = WORK_ROOT / "hf_cache"': 'HF_HOME = Path(os.environ.get("QAI_HF_HOME", str(WORK_ROOT / "hf_cache"))).expanduser().resolve()',
    'DATA_DIR = WORK_ROOT / "data"': 'DATA_DIR = Path(os.environ.get("QAI_DATA_ROOT", str(WORK_ROOT / "data"))).expanduser().resolve()',
}
runner_changed = False
for old, new in colab_optimizations.items():
    if new not in runner_source:
        if old not in runner_source:
            raise RuntimeError(f'Cannot safely apply Colab optimization; missing: {old}')
        runner_source = runner_source.replace(old, new, 1)
        runner_changed = True
if runner_changed:
    runner_path.write_text(runner_source, encoding='utf-8')
    print('Applied local Colab batching/cache optimization.')

# Keep large, frequently accessed downloads on Colab local disk.
for cache_name in ('hf_cache', 'data'):
    (LOCAL_CACHE_ROOT / cache_name).mkdir(parents=True, exist_ok=True)

for persistent_name in ('qualcomm_artifacts', 'checkpoints', 'results'):
    (DRIVE_ROOT / persistent_name).mkdir(parents=True, exist_ok=True)

print('Repository:', REPO_DIR)
print('Persistent benchmark root:', DRIVE_ROOT)
print('Fast local cache:', LOCAL_CACHE_ROOT)


In [ ]:
import shutil

VENV_DIR = Path('/content/qai_s24_venv')
VENV_PYTHON = VENV_DIR / 'bin' / 'python'

def venv_is_ready():
    if not VENV_PYTHON.exists():
        return False
    result = subprocess.run(
        [str(VENV_PYTHON), '-m', 'pip', '--version'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )
    return result.returncode == 0

if not venv_is_ready():
    if VENV_DIR.exists():
        print('Removing an incomplete virtual environment from a previous run.')
        shutil.rmtree(VENV_DIR)

    # Colab's Python 3.12 image may omit ensurepip/python3.12-venv.
    # virtualenv bundles the bootstrap needed to create an isolated pip.
    run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'virtualenv'])
    run([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(VENV_DIR)])

if not venv_is_ready():
    raise RuntimeError(f'Virtual environment was not created correctly: {VENV_DIR}')

run([str(VENV_PYTHON), '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([str(VENV_PYTHON), '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements.txt')])
run([str(VENV_PYTHON), '-c', 'import qai_hub, datasets, transformers, torch; print("Core imports: OK")'])
run([str(VENV_PYTHON), '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR)
print('Dependencies and regression preflight: OK')


In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get('QAI_HUB_API_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass('Qualcomm AI Hub API token: ').strip()
if not token:
    raise RuntimeError('Missing QAI_HUB_API_TOKEN')

os.environ['QAI_HUB_API_TOKEN'] = token
del token
print('Qualcomm token loaded securely; it will not be printed or written to output files.')


In [ ]:
os.environ["QAI_RUN_MODE"] = "benchmark"
os.environ["QAI_ARTIFACT_POLICY"] = "separate_qnn_dlc"
os.environ["QAI_HUB_JOB_RETRIES"] = "3"
os.environ["QAI_HUB_MICROBATCH"] = "4"
os.environ["QAI_ENABLE_PROFILING"] = "1"
os.environ["QAI_BENCHMARK_ROOT"] = str(DRIVE_ROOT)
os.environ["QAI_HF_HOME"] = str(LOCAL_CACHE_ROOT / 'hf_cache')
os.environ["QAI_DATA_ROOT"] = str(LOCAL_CACHE_ROOT / 'data')
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

print({
    'samples_per_benchmark': 50,
    'models': ['Whisper Tiny', 'Whisper Small', 'PhoWhisper Base'],
    'device': 'Samsung Galaxy S24',
    'compute': 'NPU',
    'microbatch': int(os.environ['QAI_HUB_MICROBATCH']),
    'persistent_root': os.environ['QAI_BENCHMARK_ROOT'],
})


In [ ]:
benchmark_env = os.environ.copy()
run(
    [str(VENV_PYTHON), '-u', 'run_benchmark.py'],
    cwd=REPO_DIR,
    env=benchmark_env,
)


In [ ]:
import pandas as pd
from IPython.display import display

FINAL_TABLE = DRIVE_ROOT / 'FINAL_BENCHMARK_TABLE.csv'
BUNDLE = DRIVE_ROOT / 'QAI_S24_BENCHMARK_SUBMISSION.zip'

if not FINAL_TABLE.exists() or not BUNDLE.exists():
    raise RuntimeError('Benchmark returned without complete final outputs')

display(pd.read_csv(FINAL_TABLE))
print('Final table:', FINAL_TABLE)
print('Submission ZIP:', BUNDLE)
print('Files are already saved safely in Google Drive.')
